# TimeSeries Data Model — In Action

This notebook demonstrates the **TimeSeries** class from `timedatamodel`: creation, inspection, slicing, arithmetic, integration with pandas/NumPy, and serialization. We use a single example (hourly power in MW) throughout.

## 1. Create a TimeSeries

Build a `TimeSeries` from a **Resolution** (frequency + timezone), scalar metadata fields, and parallel lists of timestamps and values. `None` values represent missing data.

In [ ]:
from datetime import datetime, timedelta, timezone
from timedatamodel import TimeSeries, Resolution, DataPoint, Frequency, DataType

res = Resolution(Frequency.PT1H, "UTC")

base = datetime(2024, 1, 15, tzinfo=timezone.utc)
timestamps = [base + timedelta(hours=i) for i in range(24)]
values = [100.0 + i * 2.5 for i in range(24)]
values[12] = None  # missing hour

ts = TimeSeries(res, timestamps=timestamps, values=values, name="power", unit="MW", data_type=DataType.ACTUAL)
ts

You can also build from a list of **DataPoint** named tuples:

In [ ]:
data = [
    DataPoint(datetime(2024, 1, 1, i, tzinfo=timezone.utc), 10.0 * i)
    for i in range(5)
]
ts_short = TimeSeries(res, data=data, name="power", unit="MW", data_type=DataType.ACTUAL)
list(ts_short)

## 2. Inspect and slice

Use `begin`, `end`, `duration`, `has_missing`, and the `in` operator. Indexing and slicing return **DataPoint**s; **head** and **tail** return new TimeSeries.

In [ ]:
print("Begin:", ts.begin)
print("End:", ts.end)
print("Duration:", ts.duration)
print("Has missing:", ts.has_missing)
print("Contains 12:00 UTC:", ts.begin + timedelta(hours=12) in ts)

first = ts[0]
print("\nFirst point:", first.timestamp, "->", first.value)
print("First 3 (head):", ts.head(3).values)

## 3. Scalar arithmetic

TimeSeries supports `+`, `-`, `*`, `/` with scalars, and `abs()`, `round()`. Missing values (None) are preserved as NaN in the result.

In [ ]:
scaled = ts * 0.001  # MW -> GW (conceptually)
print("First value (scaled):", scaled[0].value)

offset = ts + 10.0
print("First value (+10):", offset[0].value)

rounded = round(ts, 0)
print("Rounded [1]:", rounded[1].value)

## 4. NumPy and pandas bridges

Export to **NumPy** (`to_numpy()`) and **pandas** (`to_pandas_dataframe()`). Use **apply_numpy** and **apply_pandas** to run transformations and get back a TimeSeries.

In [ ]:
import numpy as np
import pandas as pd

arr = ts.to_numpy()
print("NumPy shape:", arr.shape, "dtype:", arr.dtype)
print("NaN at index 12:", np.isnan(arr[12]))

df = ts.to_pandas_dataframe()
print("\nDataFrame columns:", df.columns.tolist())
df.head()

In [ ]:
# Apply a numpy transformation (e.g. fill NaN with interpolated values)
filled = ts.apply_numpy(lambda a: np.where(np.isnan(a), np.nanmean(a), a))
print("After fill (index 12):", filled[12].value)

# Apply a pandas transformation (e.g. rolling mean)
smoothed = ts.apply_pandas(lambda df: df.rolling(3, min_periods=1).mean())
print("Smoothed first 3 values:", [smoothed[i].value for i in range(3)])

## 5. Round-trip from pandas

Create a TimeSeries from a DataFrame with a DatetimeIndex. Resolution can be inferred if the index has a frequency.

In [ ]:
ts_round = TimeSeries.from_pandas(df, resolution=res)
print("From pandas: length", len(ts_round), ", first:", ts_round[0])

## 6. JSON and CSV I/O

Serialize to JSON (timestamps as ISO-8601) and to CSV files; load back with **from_json** and **from_csv**.

In [ ]:
json_str = ts.to_json()
print("JSON (first 120 chars):", json_str[:120], "...")

ts_restored = TimeSeries.from_json(json_str, res)
print("Restored length:", len(ts_restored), ", match:", ts_restored[0].value == ts[0].value)

In [ ]:
from pathlib import Path

path = Path("timeseries_showcase_export.csv")
ts.to_csv(path)
ts_from_csv = TimeSeries.from_csv(path, res)
print("From CSV: length", len(ts_from_csv))
path.unlink(missing_ok=True)  # cleanup

## 7. Validation

**validate()** returns a list of warnings (e.g. non-increasing timestamps or inconsistent frequency).

In [ ]:
print("Our series:", ts.validate())

bad = TimeSeries(
    res,
    timestamps=[base + timedelta(hours=1), base, base + timedelta(hours=3)],
    values=[1.0, 2.0, 3.0],
)
print("Bad series:", bad.validate())